# 1. 数据准备和预处理

In [ ]:
import cv2
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# 1.1 定义数据增强（用opencv实现图像旋转/翻转，替代传统的torchvision）
def augment_image(image):  # 定义一个叫 augment_image的函数，它接收一个参数叫 image
    """用OpenCV实现随机水平翻转和旋转"""
    img = np.array(image)  # 把输入的图片转换成 NumPy 数组格式，方便后续处理
    if np.random.rand() > 0.5:  #  生成一个 0 到 1 之间的随机数。如果这个数大于 0.5，就执行下面的翻转操作
        img = cv2.flip(img,1)  # 对图片进行水平翻转（左右颠倒）。1代表水平翻转，0代表垂直翻转。
    if np.random.rand() > 0.7:  
        angle = np.random.uniform(-15,15)  # 生成一个在 -15 到 15 之间的随机角度。这意味着图片可能会顺时针或逆时针旋转最多 15 度。
        M = cv2.getRotationMatrix2D((img.shape[1]/2,img.shape[0]/2),angle,1.0)
        # (img.shape[1]/2, img.shape[0]/2)：这是旋转的中心点，也就是图片的正中心;angle：刚才生成的随机角度,1.0：缩放比例，1.0 代表不缩放保持原大小。
        img = cv2.warpAffine(img,M,(img.shape[1],img.shape[0]))
    return Image.fromarray(img)


# 1.2 加载CIFAR-10数据集（带自定义增强）
transform_train = transforms.Compose([
    transforms.Lambda(augment_image),
    transforms.RandomCrop(32,padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

trainset = torchvision.datasets.CIFAR10(root='./data',train=True,download=True,transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data',train=False,download=True,transform=transform_test)
trainloader = torch.utils.data.DataLoader(trainset,batch_size=128,shuffle=True,num_workers=2)
testloader = torch.utils.data.DataLoader(testset,batch_size=100,shuffle=False,num_workers=2)

# 1.3 可视化增强效果
def show_augmented_images():
    images,labels = next(iter(trainloader))
    plt.figure(figsize=(12,4))
    for i in range(6):
        plt.subplot(1,6,i+1)
        plt.imshow(np.transpose(images[i].numpy(),(1,2,0)))
        plt.title(['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'][labels[i]])
        plt.axis('off')
    plt.savefig('augmented_examples.png')  # 保存可视化
    plt.show()

show_augmented_images()  # 生成数据增强示例图

# 2. 构建ResNet18模型

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class ResNet18(nn.Module):
    def __init__(self):
        super(ResNet18,self).__init__()
        # 用torchvision预训练模型加载基础结构
        self.resnet = torchvision.models.resnet18(pretrained=False)
        # 修改输出层适配CIFAR-10的10类别
        self.resnet.fc = nn.Linear(512,10)
    def forward(self,x):
        return self.resnet(x)
model = ResNet18().to('cpu') # cpu运行

# 3.训练与评估

In [ ]:
# 3.1 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
# 3.2 训练循环
def train():
    model.train()
    train_loss = 0
    for batch_idx,(data,target) in enumerate(trainloader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output,target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    return train_loss / len(trainloader)
# 3.3 测试函数
def test():
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data,target in testloader:
            output = model(data)
            test_loss += criterion(output,target).item()
            pred = output.argmax(dim=1,keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(testloader)
    accuracy = 100.*correct / len(testloader.dataset)
    return test_loss,accuracy

# 4. 结果可视化

In [ ]:
# 4.1 在训练循环外面初始化两个空列表
train_losses = []  # 记录每轮训练损失
test_accs = []  # 记录每轮测试准确率
# 4.2 开始训练循环
for epoch in range(10):
    train_loss = train()
    test_loss,test_acc = test()
    # 4.3 等所有 epoch 结束后，把结果添加到列表里
    train_losses.append(train_loss)
    test_accs.append(test_acc)
    print(f'Epoch{epoch+1}/10 | Train Loss:{train_loss:4f} | Test Acc:{test_acc:.2f}%')
# 4.4 等所有 epoch 都跑完了，再在外面画图
plt.figure(figsize=(10,5))
plt.plot(train_losses,'b-',label='Train Loss')
plt.plot(test_accs,'r-',label='Test Acc')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title('Training Loss and Test Accuracy')
plt.legend()
plt.savefig('training_curve.png') # 保存曲线图